In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial.transform import Rotation
from pathlib import Path
import json, warnings
warnings.filterwarnings('ignore')

try:
    import cv2
    OPENCV = True
    print(f'OpenCV {cv2.__version__}')
except ImportError:
    OPENCV = False
    print('OpenCV not found — will use manual Tsai solver only')

np.set_printoptions(precision=6, suppress=True)

OpenCV 4.8.0


/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
from pathlib import Path

TF_RECORDS_DIR = Path(
    '/home/pooja/TATRO/TATRO_Program_files/TATARO_FirmwareV1.0'
    '/xarm_ros2/install/tf_recorder/TF_Records'
)

# Set to None to auto-select the most recent CSV
CSV_PATH = Path(
    '/home/pooja/TATRO/TATRO_Program_files/TATARO_FirmwareV1.0'
    '/xarm_ros2/install/tf_recorder/TF_Records/handeye_data_20260610_201226.csv'
)

# Auto-select latest file if CSV_PATH is None
if CSV_PATH is None:
    files = sorted(TF_RECORDS_DIR.glob('handeye_data_*.csv'))
    if not files:
        raise FileNotFoundError(f'No CSV found in {TF_RECORDS_DIR}')
    CSV_PATH = files[-1]

print(f'Using: {CSV_PATH}')

Using: /home/pooja/TATRO/TATRO_Program_files/TATARO_FirmwareV1.0/xarm_ros2/install/tf_recorder/TF_Records/handeye_data_20260610_201226.csv


In [3]:
df = pd.read_csv(CSV_PATH)
N  = len(df)
print(f'Samples  : {N}')
print(f'Columns  : {len(df.columns)}')
print(f'ID range : {df.sample_id.min()} → {df.sample_id.max()}')


for pfx in ['lb_l6','lb_eef','lb_tcp','cam_marker']:
    bad = ((df[f'{pfx}_T30'] != 0) | (df[f'{pfx}_T31'] != 0) |
           (df[f'{pfx}_T32'] != 0) | (df[f'{pfx}_T33'] != 1)).sum()
    status = 'OK' if bad == 0 else f'WARNING {bad} bad rows'
    print(f'  {pfx:<14} bottom-row check: {status}')

df.head(2)

Samples  : 21
Columns  : 95
ID range : 0 → 20
  lb_l6          bottom-row check: OK
  lb_eef         bottom-row check: OK
  lb_tcp         bottom-row check: OK
  cam_marker     bottom-row check: OK


,sample_id,timestamp_sec,timestamp_nanosec,lb_l6_x,lb_l6_y,lb_l6_z,lb_l6_qx,lb_l6_qy,lb_l6_qz,lb_l6_qw,...,cam_marker_T12,cam_marker_T13,cam_marker_T20,cam_marker_T21,cam_marker_T22,cam_marker_T23,cam_marker_T30,cam_marker_T31,cam_marker_T32,cam_marker_T33
0,0,1781102617,601967207,0.234941,-0.000545,0.547038,0.989818,-0.029482,0.139248,0.000283,...,0.199418,0.038894,-0.198308,0.023069,-0.979868,0.555138,0.0,0.0,0.0,1.0
1,1,1781102641,421681535,0.289441,-0.211600,0.413601,0.967312,0.123226,0.162279,-0.150961,...,0.307932,-0.010272,-0.209166,-0.400475,-0.892115,0.466906,0.0,0.0,0.0,1.0


In [5]:
#OpenCV Hand-Eye Calib

#For link_tcp

R_gripper2base = []
t_gripper2base = []

R_target2cam = []
t_target2cam = []

for _, row in df.iterrows():

    Tg = np.array([
        [row.lb_tcp_T00, row.lb_tcp_T01, row.lb_tcp_T02, row.lb_tcp_T03],
        [row.lb_tcp_T10, row.lb_tcp_T11, row.lb_tcp_T12, row.lb_tcp_T13],
        [row.lb_tcp_T20, row.lb_tcp_T21, row.lb_tcp_T22, row.lb_tcp_T23],
        [row.lb_tcp_T30, row.lb_tcp_T31, row.lb_tcp_T32, row.lb_tcp_T33]
    ])

    Tc = np.array([
        [row.cam_marker_T00, row.cam_marker_T01, row.cam_marker_T02, row.cam_marker_T03],
        [row.cam_marker_T10, row.cam_marker_T11, row.cam_marker_T12, row.cam_marker_T13],
        [row.cam_marker_T20, row.cam_marker_T21, row.cam_marker_T22, row.cam_marker_T23],
        [row.cam_marker_T30, row.cam_marker_T31, row.cam_marker_T32, row.cam_marker_T33]
    ])

    R_gripper2base.append(Tg[:3, :3])
    t_gripper2base.append(Tg[:3, 3])

    R_target2cam.append(Tc[:3, :3])
    t_target2cam.append(Tc[:3, 3])


    # print(len(R_gripper2base))
    # print(len(t_gripper2base))
    # print(len(R_target2cam))
    # print(len(t_target2cam))

In [6]:

def compute_relative_transforms(R_list, t_list):
    """Compute relative transforms between consecutive poses"""
    R_rel = []
    t_rel = []
    
    for i in range(len(R_list) - 1):
        # Relative rotation: A_i = R_i^T @ R_{i+1}
        R_rel_i = R_list[i].T @ R_list[i+1]
        R_rel.append(R_rel_i)
        
        # Relative translation: t_i = R_i^T @ (t_{i+1} - t_i)
        t_rel_i = R_list[i].T @ (t_list[i+1] - t_list[i])
        t_rel.append(t_rel_i)
    
    return R_rel, t_rel
 
A_rot, A_trans = compute_relative_transforms(R_gripper2base, t_gripper2base)
B_rot, B_trans = compute_relative_transforms(R_target2cam, t_target2cam)
 
print(f"Computed {len(A_rot)} relative motion pairs (A_i and B_i)")
 
for i, (A_i, B_i) in enumerate(zip(A_rot, B_rot)):
    print(f"\nMotion pair {i}:")
    print(f"  A_{i} (gripper relative motion) shape: {A_i.shape}")
    print(f"  B_{i} (camera relative motion) shape: {B_i.shape}")

Computed 20 relative motion pairs (A_i and B_i)

Motion pair 0:
  A_0 (gripper relative motion) shape: (3, 3)
  B_0 (camera relative motion) shape: (3, 3)

Motion pair 1:
  A_1 (gripper relative motion) shape: (3, 3)
  B_1 (camera relative motion) shape: (3, 3)

Motion pair 2:
  A_2 (gripper relative motion) shape: (3, 3)
  B_2 (camera relative motion) shape: (3, 3)

Motion pair 3:
  A_3 (gripper relative motion) shape: (3, 3)
  B_3 (camera relative motion) shape: (3, 3)

Motion pair 4:
  A_4 (gripper relative motion) shape: (3, 3)
  B_4 (camera relative motion) shape: (3, 3)

Motion pair 5:
  A_5 (gripper relative motion) shape: (3, 3)
  B_5 (camera relative motion) shape: (3, 3)

Motion pair 6:
  A_6 (gripper relative motion) shape: (3, 3)
  B_6 (camera relative motion) shape: (3, 3)

Motion pair 7:
  A_7 (gripper relative motion) shape: (3, 3)
  B_7 (camera relative motion) shape: (3, 3)

Motion pair 8:
  A_8 (gripper relative motion) shape: (3, 3)
  B_8 (camera relative motion) sha

In [7]:
print("\n" + "=" * 80)
print("STEP 3: Build Sylvester equations AX = XB")
print("=" * 80)
 
print("\nEquation: (A_i - I) R_x = R_x (I - B_i)")
print("Vectorized form using Kronecker product:")
print("  (I ⊗ (A_i - I) - (I - B_i)^T ⊗ I) vec(R_x) = 0")
 
I3 = np.eye(3)
M_list = []
 
for i, (A_i, B_i) in enumerate(zip(A_rot, B_rot)):
    # Left side: I ⊗ (A_i - I)
    M_left = np.kron(I3, A_i - I3)
    
    # Right side: (I - B_i)^T ⊗ I
    M_right = np.kron((I3 - B_i).T, I3)
    
    # Combine: (Left - Right) vec(R_x) = 0
    M_i = M_left - M_right
    M_list.append(M_i)
    
    print(f"\nEquation {i}: M_{i} shape = {M_i.shape}")
 
# Stack all equations
M = np.vstack(M_list)
 
print(f"\n" + "-" * 80)
print(f"STACKED MATRIX M:")
print(f"  Shape: {M.shape}")
print(f"  Rows: {M.shape[0]} (total equations)")
print(f"  Cols: {M.shape[1]} (unknowns = 9 for 3×3 rotation)")
print(f"  System is {'OVERDETERMINED' if M.shape[0] > M.shape[1] else 'UNDERDETERMINED'}")


STEP 3: Build Sylvester equations AX = XB

Equation: (A_i - I) R_x = R_x (I - B_i)
Vectorized form using Kronecker product:
  (I ⊗ (A_i - I) - (I - B_i)^T ⊗ I) vec(R_x) = 0

Equation 0: M_0 shape = (9, 9)

Equation 1: M_1 shape = (9, 9)

Equation 2: M_2 shape = (9, 9)

Equation 3: M_3 shape = (9, 9)

Equation 4: M_4 shape = (9, 9)

Equation 5: M_5 shape = (9, 9)

Equation 6: M_6 shape = (9, 9)

Equation 7: M_7 shape = (9, 9)

Equation 8: M_8 shape = (9, 9)

Equation 9: M_9 shape = (9, 9)

Equation 10: M_10 shape = (9, 9)

Equation 11: M_11 shape = (9, 9)

Equation 12: M_12 shape = (9, 9)

Equation 13: M_13 shape = (9, 9)

Equation 14: M_14 shape = (9, 9)

Equation 15: M_15 shape = (9, 9)

Equation 16: M_16 shape = (9, 9)

Equation 17: M_17 shape = (9, 9)

Equation 18: M_18 shape = (9, 9)

Equation 19: M_19 shape = (9, 9)

--------------------------------------------------------------------------------
STACKED MATRIX M:
  Shape: (180, 9)
  Rows: 180 (total equations)
  Cols: 9 (unknown

In [8]:
print("\n" + "=" * 80)
print("STEP 4: SVD Decomposition M = U Σ V^T")
print("=" * 80)
 
print("\nComputing SVD...")
U, sigma, Vt = np.linalg.svd(M, full_matrices=True)
V = Vt.T
 
print(f"\nSVD Results:")
print(f"  U shape:  {U.shape}")
print(f"  Σ shape:  {sigma.shape}")
print(f"  V^T shape: {Vt.shape}")
print(f"  V shape:  {V.shape}")
 
print(f"\nSingular values (Σ):")
print(f"  {sigma}")
 
print(f"\nSmallest singular value: {sigma[-1]:.4e}")
print(f"Largest singular value:  {sigma[0]:.4e}")
print(f"Condition number: {sigma[0] / sigma[-1]:.4e}")


STEP 4: SVD Decomposition M = U Σ V^T

Computing SVD...

SVD Results:
  U shape:  (180, 180)
  Σ shape:  (9,)
  V^T shape: (9, 9)
  V shape:  (9, 9)

Singular values (Σ):
  [6.201155 6.168535 4.677394 4.497178 4.198318 4.048355 3.737573 3.435137
 2.849765]

Smallest singular value: 2.8498e+00
Largest singular value:  6.2012e+00
Condition number: 2.1760e+00


In [9]:
print("\n" + "=" * 80)
print("STEP 5: Extract solution from null space")
print("=" * 80)
 
print("\nThe solution is the column of V corresponding to the smallest singular value")
print("This is the LAST column of V (V^T last row)")
 
# Get the eigenvector for the smallest singular value (last column of V)
vec_Rx = V[:, -1]
 
print(f"\nSolution vector (vec_R_x): {vec_Rx}")
 
# Reshape to 3x3 matrix
R_svd = vec_Rx.reshape((3, 3))
 
print(f"\nRotation matrix (before orthonormalization):")
print(R_svd)
 


STEP 5: Extract solution from null space

The solution is the column of V corresponding to the smallest singular value
This is the LAST column of V (V^T last row)

Solution vector (vec_R_x): [-0.214279  0.02713  -0.517966 -0.037101  0.248725  0.004778 -0.015566
  0.084087 -0.78389 ]

Rotation matrix (before orthonormalization):
[[-0.214279  0.02713  -0.517966]
 [-0.037101  0.248725  0.004778]
 [-0.015566  0.084087 -0.78389 ]]


In [10]:
 
print("\n" + "=" * 80)
print("STEP 6: Enforce orthogonality (Procrustes problem)")
print("=" * 80)
 
print("\nThe vectorized solution might not be perfectly orthogonal.")
print("Use SVD to project onto the nearest orthogonal matrix:")
print("  U_ortho, _, V_ortho = SVD(R_svd)")
print("  R_x_ortho = U_ortho @ V_ortho^T")
 
U_ortho, _, Vt_ortho = np.linalg.svd(R_svd)
R_svd_ortho = U_ortho @ Vt_ortho
 
print(f"\nRotation matrix (after orthonormalization):")
print(R_svd_ortho)
 
# Check orthogonality
det_R = np.linalg.det(R_svd_ortho)
print(f"\ndet(R) = {det_R:.6f} (should be 1.0 or -1.0)")
 
is_orthogonal = np.allclose(R_svd_ortho @ R_svd_ortho.T, np.eye(3))
print(f"Is orthogonal: {is_orthogonal} (R @ R^T = I)")


STEP 6: Enforce orthogonality (Procrustes problem)

The vectorized solution might not be perfectly orthogonal.
Use SVD to project onto the nearest orthogonal matrix:
  U_ortho, _, V_ortho = SVD(R_svd)
  R_x_ortho = U_ortho @ V_ortho^T

Rotation matrix (after orthonormalization):
[[-0.884707 -0.082747 -0.458745]
 [-0.138729  0.986265  0.089645]
 [ 0.445026  0.142951 -0.884035]]

det(R) = 1.000000 (should be 1.0 or -1.0)
Is orthogonal: True (R @ R^T = I)


In [20]:
 
print("\n" + "=" * 80)
print("STEP 7: Solve for translation using least squares")
print("=" * 80)
 
print("\nSystem: (A_i - I) t_x = R_x B_i_trans - A_i_trans")
 
A_t_list = []
b_t_list = []
 
for i, (A_i, A_i_trans, B_i_trans) in enumerate(zip(A_rot, A_trans, B_trans)):
    # Build: (A_i - I) t_x = R_x @ B_i_trans - A_i_trans
    A_sys = A_i - I3
    b_sys = (R_svd_ortho @ B_i_trans - A_i_trans).reshape(-1, 1)
    
    A_t_list.append(A_sys)
    b_t_list.append(b_sys)
    
    print(f"Equation {i}: A shape = {A_sys.shape}, b shape = {b_sys.shape}")
 
# Stack all translation equations
A_t = np.vstack(A_t_list)
b_t = np.vstack(b_t_list)

assert A_t.shape[0] == b_t.shape[0], f"Shape mismatch: {A_t.shape} vs {b_t.shape}"

print(f"\nStacked system shape: {A_t.shape}")
 
t_svd, residuals, rank, s = np.linalg.lstsq(A_t, b_t, rcond=None)
 
print(f"\nLeast squares solution:")
print(f"  Translation: {t_svd.flatten()}")
print(f"  Residuals: {residuals}")
print(f"  Rank: {rank}")
 


STEP 7: Solve for translation using least squares

System: (A_i - I) t_x = R_x B_i_trans - A_i_trans
Equation 0: A shape = (3, 3), b shape = (3, 1)
Equation 1: A shape = (3, 3), b shape = (3, 1)
Equation 2: A shape = (3, 3), b shape = (3, 1)
Equation 3: A shape = (3, 3), b shape = (3, 1)
Equation 4: A shape = (3, 3), b shape = (3, 1)
Equation 5: A shape = (3, 3), b shape = (3, 1)
Equation 6: A shape = (3, 3), b shape = (3, 1)
Equation 7: A shape = (3, 3), b shape = (3, 1)
Equation 8: A shape = (3, 3), b shape = (3, 1)
Equation 9: A shape = (3, 3), b shape = (3, 1)
Equation 10: A shape = (3, 3), b shape = (3, 1)
Equation 11: A shape = (3, 3), b shape = (3, 1)
Equation 12: A shape = (3, 3), b shape = (3, 1)
Equation 13: A shape = (3, 3), b shape = (3, 1)
Equation 14: A shape = (3, 3), b shape = (3, 1)
Equation 15: A shape = (3, 3), b shape = (3, 1)
Equation 16: A shape = (3, 3), b shape = (3, 1)
Equation 17: A shape = (3, 3), b shape = (3, 1)
Equation 18: A shape = (3, 3), b shape = (3,

In [21]:
print("\n" + "=" * 80)
print("FINAL RESULTS: Hand-Eye Transformation")
print("=" * 80)
 
print("\nROTATION MATRIX (3×3):")
print(R_svd_ortho)
 
print("\nTRANSLATION VECTOR (3×1):")
print(t_svd)
 
print("\nCombined 4×4 Transformation Matrix:")
T_hand2eye = np.eye(4)
T_hand2eye[:3, :3] = R_svd_ortho
T_hand2eye[:3, 3:4] = t_svd
print(T_hand2eye)


FINAL RESULTS: Hand-Eye Transformation

ROTATION MATRIX (3×3):
[[-0.884707 -0.082747 -0.458745]
 [-0.138729  0.986265  0.089645]
 [ 0.445026  0.142951 -0.884035]]

TRANSLATION VECTOR (3×1):
[[-0.026791]
 [ 0.012197]
 [-0.230594]]

Combined 4×4 Transformation Matrix:
[[-0.884707 -0.082747 -0.458745 -0.026791]
 [-0.138729  0.986265  0.089645  0.012197]
 [ 0.445026  0.142951 -0.884035 -0.230594]
 [ 0.        0.        0.        1.      ]]
